In [55]:
from utils import getDevice, collate_fn #important to impot at the start for reproducibility :3 

In [56]:
# GENERAL IMPORTS
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import time
import torch
import gc
import os
import pandas as pd
from IPython.display import clear_output

from torchvision.models import vgg16
from torchvision import models
from torchvision import transforms
from torchvision.ops import roi_pool
import torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split

# CUSTIM FUNCTIONS AND VARIABLES
from kitty import depth_read, listPicsWith, togglePath, MatchDepthToCar, KITTY_PATH
from yolo import getEmbedFromResults, getCropsFromResults, getCropsFromResult, getEmbedFromCrops, CLASSES_YOLO, CONFIDENCE_YOLO

# YOLO STUFF
from ultralytics import YOLO

#DINO STUFF
from transformers import AutoImageProcessor, AutoModel
from transformers.image_utils import load_image

In [57]:
DEVICE = getDevice()
HEIGHT = 376
WIDTH = 1242
TARGET_TYPES = ['Car']
BATCH_SIZE = 4
NUM_WORKERS = 4

In [58]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((HEIGHT, WIDTH))
])

#DATASET IS LOADED AND DATALOADER IS CREATED

kitty_dataset =torchvision.datasets.Kitti(root="../datasets/", train=True,transform=transform, download=True)
kittty_dataloader = torch.utils.data.DataLoader(
    kitty_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn, 
    num_workers=NUM_WORKERS,
    pin_memory=True
)

KITTY_LENGTH = len(kitty_dataset)
BATCH_COUNT = KITTY_LENGTH // BATCH_SIZE + (KITTY_LENGTH % BATCH_SIZE > 0)
print(f"KITTY dataset is length: {KITTY_LENGTH}\nDataloader has length: {len(kittty_dataloader)}\nBatch count {BATCH_COUNT}")

#I try a single batch to analyze the data structure and types
images, targets_batch = next(iter(kittty_dataloader))
print(f"Type of the images data is: {type(images)} type of image {type(images[0])} len: {len(images)} and type of the target is {type(targets_batch)} of len {len(targets_batch)}")

KITTY dataset is length: 7481
Dataloader has length: 1871
Batch count 1871
Type of the images data is: <class 'list'> type of image <class 'torch.Tensor'> len: 4 and type of the target is <class 'list'> of len 4


In [59]:
for i, targets in enumerate(targets_batch):
    print(f"Target type is {type(targets)} and length is {len(targets)}")
    for target in targets:
        print(type(target))  # Should be a dict
        obj_types  = target["type"]        # e.g. ['Car', 'Pedestrian']
        locations  = target["location"]    # Tensor [N, 3]: (X, Y, Z) in metres
        bboxes     = torch.Tensor(target["bbox"])        # Tensor [N, 4]: (x1, y1, x2, y2) pixels
        print(f"Object types: {obj_types} of type {type(obj_types)}")
        print(f"Locations: {locations} of type {type(locations)}")
        print(f"Bounding boxes: {bboxes} of type {type(bboxes)}")

    print("")

Target type is <class 'list'> and length is 1
<class 'dict'>
Object types: Pedestrian of type <class 'str'>
Locations: [-3.68, 1.47, 9.0] of type <class 'list'>
Bounding boxes: tensor([275.0000, 136.7000, 353.9000, 299.5400]) of type <class 'torch.Tensor'>

Target type is <class 'list'> and length is 8
<class 'dict'>
Object types: Car of type <class 'str'>
Locations: [-3.17, 1.8, 6.11] of type <class 'list'>
Bounding boxes: tensor([  0.0000, 188.1200, 401.1400, 374.0000]) of type <class 'torch.Tensor'>
<class 'dict'>
Object types: Car of type <class 'str'>
Locations: [2.93, 1.65, 6.84] of type <class 'list'>
Bounding boxes: tensor([ 785.1800,  184.5000, 1153.9600,  374.0000]) of type <class 'torch.Tensor'>
<class 'dict'>
Object types: Truck of type <class 'str'>
Locations: [-3.33, 1.73, 15.75] of type <class 'list'>
Bounding boxes: tensor([371.5800, 122.4100, 520.8200, 267.4900]) of type <class 'torch.Tensor'>
<class 'dict'>
Object types: Car of type <class 'str'>
Locations: [2.77, 1.7

In [60]:
yolo_embeds_paths =[]
dino_embeds_paths =[]
dino_box_embeds_paths =[]

In [61]:
DINO_MODEL_ID = "facebook/dinov3-vits16-pretrain-lvd1689m"

image_processor = AutoImageProcessor.from_pretrained(DINO_MODEL_ID)
dinov3 = AutoModel.from_pretrained(
    DINO_MODEL_ID,
    dtype=torch.float16,
    device_map="auto"
).to(DEVICE)

def DINOgetEmbed(image):
    """
    Get DINOv3 embeddings from an image or a list of images. 
    Args:
        image: string path or PIL image or list of ready to pricess opened imaged
    """
    #case 1 - image is a path
    if type(image) == str:
        image = load_image(image)
    
    #case 2 is pil image already or compatible with the class
    inputs = image_processor(images=image, return_tensors="pt").to(DEVICE)

    #retrieve embeddings
    dinov3.eval()
    with torch.inference_mode():
        outputs = dinov3(**inputs)

    return outputs.pooler_output  # final 
    
def DINOgetEmbedBoxes(image, boxes):
    """
    Extract embeddings from a single image using a list of boxes in that image. The boxes
    should be in dimensions of the image.

    Args:
        image: image to process either str or pil
        boxes: list of boxes to process
    """
    embeds = []

    #case 1 - image is a path
    if type(image) == str:
        image = load_image(image)

    #case 2 is pil image already or compatible with the class
    inputs = image_processor(images=image, return_tensors="pt").to(DEVICE)

    #this is based on the official tutorial (http://medium.com/@davidrustsmith/dino-v3-with-huggingface-basics-8f9630943ea2)
    batch_size, _, img_height, img_width = inputs.pixel_values.shape
    print(f"image shape is {image.shape}")
    _, orig_height, orig_width= image.shape
    
    patch_size = dinov3.config.patch_size
    num_patches_height, num_patches_width = img_height // patch_size, img_width // patch_size

    #retrieve embeddings
    dinov3.eval()
    with torch.inference_mode():
        outputs = dinov3(**inputs)

    last_hidden_states = outputs.last_hidden_state

    #patch features are gathered here today to celebrate
    #cls_token = last_hidden_states[:, 0, :]
    patch_features_flat = last_hidden_states[:, 1 + dinov3.config.num_register_tokens:, :]
    patch_features = patch_features_flat.unflatten(1, (num_patches_height, num_patches_width))

    #now we can takes the boxes of the image and use it to our advantage to extract the embeddings
    for box in boxes:
        x1,y1,x2,y2 = box.tolist()
        #the dimensions are scaled to fit the new size
        x1s = int(x1/orig_width * num_patches_width)
        x2s = int(x2/orig_width * num_patches_width)
        y1s = int(y1/orig_height * num_patches_height)
        y2s = int(y2 / orig_height * num_patches_height)

        x1s = max(0, x1s)
        y1s = max(0, y1s)

        x2s = min(x2s, num_patches_width)
        y2s = min(y2s, num_patches_height)

        embed = patch_features[0, y1s:y2s, x1s:x2s, :].mean(dim=(0,1))
        embeds.append(embed)

    del outputs, patch_features_flat, patch_features
    return embeds


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

In [73]:
def extract_crops_depth(images_batch, targets_batch):
    # 1 IMAGES
        stacked_images = torch.stack(images_batch)
        # 2,3 BOXES and DEPTHS
        boxes =[]
        depths = []
        skip = []
        image_indicies = set(range(len(images_batch)))
        keep=[]
        crops=[]
        for i, targets in enumerate(targets_batch):
            boxes_with_target = [torch.tensor(target["bbox"], dtype=torch.int32) for target in targets if target["type"] in TARGET_TYPES]
            depths_with_target = [target["location"][2] for target in targets if target["type"] in TARGET_TYPES]
            if len(boxes_with_target) > 0:
                boxes.append(torch.stack(boxes_with_target))
                depths.append(torch.Tensor(depths_with_target))
            else:
                skip.append(i)

        # 3.5 check that there is actually data 
        if len(boxes) == 0:
            print("No boxes found in this batch, skipping...")
            del boxes, images_batch, targets_batch
            gc.collect()
            return -1, -1,-1,-1

        #3.6 skip images that we do not want and add the depths
        keep = list(image_indicies - set(skip))
        stacked_images = stacked_images[keep,:,:,:]
        depths = torch.cat(depths).cpu()
        #depths_true_all.extend(depths.tolist())

        # 4 crop the images to a specific size
        for i in range(stacked_images.shape[0]):
            # the current data is fetched
            image = stacked_images[i]
            boxes_cur = boxes[i]

            #images are copped to the box
            for box in boxes_cur:
                x1,y1,x2,y2 = box.tolist()
                crop_tensor = image[:, y1:y2, x1:x2]
                crop_np = crop_tensor.permute(1, 2, 0).numpy()
                crop_np = (crop_np * 255).clip(0, 255).astype(np.uint8)
                crops.append(crop_np)

        #exit function and clear the local variables
        del targets_batch
        return crops, depths.tolist(), boxes, stacked_images

def extract_crops(loader: DataLoader):
    """
    Extract the crops before the embeddings are retrieved.

    Args:
        loader: dataloader for the dataset

    Returns:
        depths: list of all depths in the images
    """
    depths_true_all = []
    counter = 0
    chunk_yolo=[]
    chunk_dino=[]
    chunk_dino_box=[]
    chunk_count=0
    
    for images_batch, targets_batch in loader:
        #preporcess the data, format it in a way that is expected by the model===================
        clear_output(wait=True)
        print(f"Batch {counter} in progress; ({counter/len(loader)*100:.2f}%);")

        #extract boxes and depths to be added and saved==========================================
        crops, depths, boxes, stacked_images = extract_crops_depth(images_batch, targets_batch)
        if crops==-1 and depths==-1:
            continue
        depths_true_all.extend(depths)

        # DINO box embeddings using 1 image pass through=========================================
        for i, box_img in enumerate(boxes):
            image = stacked_images[i]
            em_dino_box = DINOgetEmbedBoxes(image, box_img)
            em_dino_box = torch.stack(em_dino_box)
            print(f"embed shape is {em_dino_box.shape}")
            chunk_dino_box.append(em_dino_box)
            del em_dino_box

        #YOLO, DINO embeds=======================================================================
        for crop in crops:
            em_yolo = getEmbedFromCrops(crop)
            em_dino = DINOgetEmbed(crop)
            chunk_yolo.append(em_yolo[0])
            chunk_dino.append(em_dino[0])

        #flush the batch into the memory
        chumky_path_yolo = f"../data/yolo_embeds_chunk_{chunk_count}.pt"
        chumky_path_dino = f"../data/dino_embeds_chunk_{chunk_count}.pt"
        chumky_path_dino_box = f"../data/dino_box_embeds_chunk_{chunk_count}.pt"
        torch.save(torch.stack(chunk_yolo), chumky_path_yolo)
        torch.save(torch.stack(chunk_dino), chumky_path_dino)
        torch.save(torch.cat(chunk_dino_box), chumky_path_dino_box)
        #save the path
        yolo_embeds_paths.append(chumky_path_yolo)   
        dino_embeds_paths.append(chumky_path_dino)     
        dino_box_embeds_paths.append(chumky_path_dino_box)

        #clear the variables and incremenet the counter==========================================
        del crops, depths, chunk_yolo, chunk_dino, chunk_dino_box, boxes, stacked_images
        chunk_yolo = []
        chunk_dino = []
        chunk_dino_box = []
        chunk_count += 1
        counter+=1

    gc.collect() 
    return depths_true_all

"""
-----------------------------------------------------------------------------------------------
SEPARATE FUNCTIONS FOR EACH METHOD
this is useful if only one of them needs to be used or for tracking time. The function above is 
way more efficient as it only passed through the data once, but it also obsucres the amount of time
it takes per image for different models.
-----------------------------------------------------------------------------------------------
"""
def extract_crops_dino(loader: DataLoader, save_data=False):
    """
    Extract the crops before the embeddings are retrieved.

    Args:
        loader: dataloader for the dataset
        save_data: False whether to save the data to a file

    Returns:
        time: time it took to run
        depths: list of all depths in the images
    """
    start = time.time()

    depths_true_all = []
    counter = 0
    chunk_dino=[]
    chunk_count=0
    
    for images_batch, targets_batch in loader:
        #preporcess the data, format it in a way that is expected by the model===================
        clear_output(wait=True)
        print(f"Batch {counter} in progress; ({counter/len(loader)*100:.2f}%);")

        #extract boxes and depths to be added and saved==========================================
        crops, depths, _, _ = extract_crops_depth(images_batch, targets_batch)
        if crops==-1 and depths==-1:
            continue
        depths_true_all.extend(depths)

        #YOLO embeds=============================================================================
        #embeds = []
        for crop in crops:            
            em_dino = DINOgetEmbed(crop)
            #print(f"the type of em dino is {type(em_dino)} of shape{em_dino.shape} the item added is {em_dino[0].shape}")
            chunk_dino.append(em_dino[0])

        #flush the batch into the memory
        chumky_path_dino = f"../data/dino_embeds_chunk_{chunk_count}.pt" 
        if save_data:
            torch.save(torch.stack(chunk_dino), chumky_path_dino)   
            dino_embeds_paths.append(chumky_path_dino) 

        #clear the variables and incremenet the counter===========================================
        del crops, depths, chunk_dino
        chunk_dino = []
        chunk_count += 1
        counter+=1

    gc.collect() 

    end = time.time()
    return (end-start), depths_true_all

def extract_crops_yolo(loader: DataLoader, save_data=False):
    """
    Extract the crops before the embeddings are retrieved.

    Args:
        loader: dataloader for the dataset
        save_data: False whether to save the data to a file
    
    Returns:
        time: time it took to run
        depths: list of all depths in the images
    """
    start = time.time()
    depths_true_all = []
    counter = 0
    chunk_yolo=[]
    chunk_count=0
    
    for images_batch, targets_batch in loader:
        #preporcess the data, format it in a way that is expected by the model===================
        clear_output(wait=True)
        print(f"Batch {counter} in progress; ({counter/len(loader)*100:.2f}%);")

        #extract boxes and depths to be added and saved==========================================
        crops, depths, _, _ = extract_crops_depth(images_batch, targets_batch)
        if crops==-1 and depths==-1:
            continue
        depths_true_all.extend(depths)

        #YOLO embeds=============================================================================
        #embeds = []
        for crop in crops:
            em_yolo = getEmbedFromCrops(crop)
            #print(f"the type of em dino is {type(em_dino)} of shape{em_dino.shape} the item added is {em_dino[0].shape}")
            chunk_yolo.append(em_yolo[0])

        #flush the batch into the memory
        chumky_path_yolo = f"../data/yolo_embeds_chunk_{chunk_count}.pt"   
        if save_data:
            torch.save(torch.stack(chunk_yolo), chumky_path_yolo)
            yolo_embeds_paths.append(chumky_path_yolo)   

        #clear the variables and incremenet the counter===========================================
        del crops, depths, chunk_yolo
        chunk_yolo = []
        chunk_count += 1
        counter+=1

    gc.collect() 
    end = time.time()
    return (end-start), depths_true_all

def extract_crops_dino_box(loader: DataLoader, save_data=False):
    """
    Extract the crops before the embeddings are retrieved.

    Args:
        loader: dataloader for the dataset
        save_data: False whether to save the data to a file

    Returns:
        time: time it took to run
        depths: list of all depths in the images
    """
    start = time.time()
    depths_true_all = []
    counter = 0
    chunk_dino_box=[]
    chunk_count=0
    
    for images_batch, targets_batch in loader:
        #preporcess the data, format it in a way that is expected by the model===================
        clear_output(wait=True)
        print(f"Batch {counter} in progress; ({counter/len(loader)*100:.2f}%);")

        #extract boxes and depths to be added and saved==========================================
        crops, depths, boxes, stacked_images = extract_crops_depth(images_batch, targets_batch)
        if crops==-1 and depths==-1:
            continue
        depths_true_all.extend(depths)

        # DINO box embeddings using 1 image pass through=========================================
        for i, box_img in enumerate(boxes):
            image = stacked_images[i]
            em_dino_box = DINOgetEmbedBoxes(image, box_img)
            em_dino_box = torch.stack(em_dino_box)
            print(f"embed shape is {em_dino_box.shape}")
            chunk_dino_box.append(em_dino_box)
            del em_dino_box


        #flush the batch into the memory
        chumky_path_dino_box = f"../data/dino_box_embeds_chunk_{chunk_count}.pt"
        if save_data:
            torch.save(torch.cat(chunk_dino_box), chumky_path_dino_box)  
            dino_embeds_paths.append(chumky_path_dino_box)  
        #clear the variables and incremenet the counter==========================================
        del crops, depths, boxes, stacked_images, chunk_dino_box
        chunk_dino_box = []
        chunk_count += 1
        counter+=1

    gc.collect() 
    end=time.time()
    return (end-start), depths_true_all


In [63]:
depths_true = extract_crops(kittty_dataloader)

Batch 1870 in progress; (99.95%);
image shape is torch.Size([3, 376, 1242])
embed shape is torch.Size([1, 384])



In [64]:
print("Merging DINO embedding chunks...")
all_chunks = [torch.load(p) for p in dino_embeds_paths]
full_embeds = torch.cat(all_chunks, dim=0)
torch.save(full_embeds, "../data/embeds_dino_og_boxes.pt")
print(f"Saved DINO embeddings: {full_embeds.shape}")
del all_chunks, full_embeds
gc.collect()

for path in dino_embeds_paths:
    os.remove(path)

Merging DINO embedding chunks...
Saved DINO embeddings: torch.Size([28742, 384])


In [65]:
print("Merging YOLO embedding chunks...")
all_chunks = [torch.load(p) for p in yolo_embeds_paths]
full_embeds = torch.cat(all_chunks, dim=0)
torch.save(full_embeds, "../data/embeds_yolo_og_boxes.pt")
print(f"Saved YOLO embeddings: {full_embeds.shape}")
del all_chunks, full_embeds
gc.collect()

for path in yolo_embeds_paths:
    os.remove(path)

Merging YOLO embedding chunks...
Saved YOLO embeddings: torch.Size([28742, 256])


In [68]:
print("Merging DINO box embedding chunks...")
all_chunks = [torch.load(p) for p in dino_box_embeds_paths]
full_embeds = torch.cat(all_chunks, dim=0)
torch.save(full_embeds, "../data/embeds_dino_box_og_boxes.pt")
print(f"Saved DINO box embeddings: {full_embeds.shape}")
del all_chunks, full_embeds
gc.collect()

for path in dino_box_embeds_paths:
    os.remove(path)

Merging DINO box embedding chunks...
Saved DINO box embeddings: torch.Size([28742, 384])


In [66]:
depths_tensor = torch.Tensor(depths_true)
torch.save(depths_tensor, "../data/true_depth.pt")
print(f"Saved the true depths into a file with shape: {depths_tensor.shape}")
del depths_tensor
gc.collect()

Saved the true depths into a file with shape: torch.Size([28742])


16

## Evaluate time per image

In [74]:
time_dino,_ = extract_crops_dino(kittty_dataloader)

Batch 1870 in progress; (99.95%);


In [75]:
time_dino_box,_ = extract_crops_dino_box(kittty_dataloader)

Batch 1870 in progress; (99.95%);
image shape is torch.Size([3, 376, 1242])
embed shape is torch.Size([7, 384])


In [76]:
time_yolo,_ = extract_crops_yolo(kittty_dataloader)

Batch 1869 in progress; (99.89%);





Values recorded last time when the code was run:

```
YOLO time per image embedding is 0.07410672743233461
DINO time per image embedding is 0.09074155246330254
DINO box time per image embedding is 0.028977343430893258
```

In [77]:
print(f"YOLO time per image embedding is {time_yolo/KITTY_LENGTH}")
print(f"DINO time per image embedding is {time_dino/KITTY_LENGTH}")
print(f"DINO box time per image embedding is {time_dino_box/KITTY_LENGTH}")

YOLO time per image embedding is 0.07410672743233461
DINO time per image embedding is 0.09074155246330254
DINO box time per image embedding is 0.028977343430893258
